# ⚖️ Comparativa de Modelos: Base vs. Fine-Tuned

Este notebook permite evaluar visualmente las diferencias en las respuestas entre el modelo original (base) y el modelo entrenado con tus datos específicos.

In [5]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

# Cargar configuración desde el archivo .env en la raíz
load_dotenv(dotenv_path="../.env")

api_key = os.getenv("AZURE_OPENAI_KEY") or os.getenv("KEY")
base_url = os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("BASE_URL")
base_model = os.getenv("BASE_MODEL")
ft_model = os.getenv("FINETUNED_MODEL")

if not api_key or not base_url:
    print("❌ Error: No se encontraron las credenciales en el archivo .env")
else:
    # Usamos el cliente estándar de OpenAI para máxima compatibilidad con Foundry Project API
    client = OpenAI(
        api_key=api_key,
        base_url=base_url
    )
    print(f"📌 Modelo Base: {base_model}")
    print(f"📌 Modelo Fine-Tuned: {ft_model}")
    print(f"✅ Cliente configurado en: {base_url}")

📌 Modelo Base: gpt-4o-mini
📌 Modelo Fine-Tuned: gpt-4o-mini-2024-07-18.ft-647e284ee3024ba59af17852a8377e
✅ Cliente configurado en: https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1


## 1. Preparación de Casos de Prueba
Vamos a extraer algunos ejemplos del set de validación para ver cómo se comportan los modelos.

In [6]:
def get_test_cases(file_path, num_samples=3):
    test_cases = []
    if not os.path.exists(file_path):
        print(f"❌ Error: No se encuentra el archivo {file_path}")
        return []
        
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= num_samples: break
            data = json.loads(line)
            # Extraemos el mensaje del sistema y el del usuario
            system_msg = data['messages'][0]['content']
            user_msg = data['messages'][1]['content']
            expected = data['messages'][2]['content']
            test_cases.append({"system": system_msg, "user": user_msg, "expected": expected})
    return test_cases

test_cases = get_test_cases("validation_set.jsonl")
if test_cases:
    print(f"✅ {len(test_cases)} casos cargados para la comparativa.")

✅ 3 casos cargados para la comparativa.


## 2. Ejecución de la Comparativa
Enviaremos las mismas preguntas a ambos modelos y guardaremos las respuestas.

In [7]:
results = []

if not test_cases:
    print("❌ No hay casos de prueba para ejecutar.")
else:
    for case in test_cases:
        # Respuesta Modelo Base
        try:
            print(f"Consultando Modelo Base: {base_model}...")
            resp_base = client.chat.completions.create(
                model=base_model,
                messages=[{"role": "system", "content": case['system']}, {"role": "user", "content": case['user']}],
                max_tokens=60
            ).choices[0].message.content
        except Exception as e: resp_base = f"Error: {e}"

        # Respuesta Modelo Fine-Tuned
        try:
            print(f"Consultando Modelo Fine-Tuned: {ft_model}...")
            resp_ft = client.chat.completions.create(
                model=ft_model,
                messages=[{"role": "system", "content": case['system']}, {"role": "user", "content": case['user']}],
                max_tokens=60
            ).choices[0].message.content
        except Exception as e: resp_ft = f"Error: {e}"

        results.append({
            "Input": case['user'],
            "Base Model Response": resp_base,
            "Fine-Tuned Response": resp_ft,
            "Expected (Ground Truth)": case['expected']
        })

    df = pd.DataFrame(results)
    display(HTML("<h3>Resultados de la Comparativa</h3>"))
    display(df)

Consultando Modelo Base: gpt-4o-mini...
Consultando Modelo Fine-Tuned: gpt-4o-mini-2024-07-18.ft-647e284ee3024ba59af17852a8377e...
Consultando Modelo Base: gpt-4o-mini...
Consultando Modelo Fine-Tuned: gpt-4o-mini-2024-07-18.ft-647e284ee3024ba59af17852a8377e...
Consultando Modelo Base: gpt-4o-mini...
Consultando Modelo Fine-Tuned: gpt-4o-mini-2024-07-18.ft-647e284ee3024ba59af17852a8377e...


,Input,Base Model Response,Fine-Tuned Response,Expected (Ground Truth)
0,Analiza mi perfil académico con estos datos: H...,Analizando tu perfil académico con los siguien...,Error: Error code: 404 - {'error': {'type': 'i...,"Tras analizar tus métricas, mi predicción basa..."
1,Analiza mi perfil académico con estos datos: H...,Para evaluar tu probabilidad de éxito académic...,Error: Error code: 404 - {'error': {'type': 'i...,"Tras analizar tus métricas, mi predicción basa..."
2,Analiza mi perfil académico con estos datos: H...,Para analizar tu perfil académico y predecir s...,Error: Error code: 404 - {'error': {'type': 'i...,"Tras analizar tus métricas, mi predicción basa..."


## 3. Conclusión
Observa si el modelo Fine-Tuned sigue mejor el formato esperado o si sus predicciones son más precisas comparadas con el *Expected* (Ground Truth).